In [3]:
import pandas as pd
import numpy as np
import os, json
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

# --------------------------
# CONFIG
# --------------------------

BASE = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset"
OUT_PATH = r"C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Models\Churn"
IN_CSV = os.path.join(BASE, "synthetic_prophet_dataset_v2b.csv")
OUT_DIR = os.path.join(OUT_PATH, "models_churn_v2b")
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading:", IN_CSV)
df = pd.read_csv(IN_CSV, parse_dates=["anchor_time"])

print(df.shape)
print(df.head())

# -----------------------------------------
# 1. CREATE CHURN LABEL (30-DAY)
# -----------------------------------------

if "purchase_next_30d" in df.columns:
    df["churn_30d"] = 1 - df["purchase_next_30d"].astype(int)
elif "purchase_next_30d_true" in df.columns:
    df["churn_30d"] = 1 - df["purchase_next_30d_true"].astype(int)
else:
    raise ValueError("Dataset missing purchase_next_30d or *_true — cannot derive churn.")

print("Churn rate:", df["churn_30d"].mean())

# -----------------------------------------
# 2. FEATURE SELECTION (no leakage)
# -----------------------------------------

forbidden = [c for c in df.columns if "purchase_next" in c or "future" in c]
print("Removing leakage columns:", forbidden)

feature_cols = [c for c in df.columns 
                if c not in forbidden 
                and c not in ["person", "anchor_time", "churn_30d"]]

print("Using", len(feature_cols), "features")

# -----------------------------------------
# 3. ENCODE CATEGORICALS
# -----------------------------------------

X = df[feature_cols].copy()
y = df["churn_30d"].astype(int)
groups = df["person"].astype(str)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

encoders = {}
for c in cat_cols:
    X[c] = X[c].fillna("missing").astype(str)
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c])
    encoders[c] = le

joblib.dump(encoders, os.path.join(OUT_DIR, "label_encoders.pkl"))

# -----------------------------------------
# 4. TIME-BASED SPLIT (final test set)
# -----------------------------------------

df_sorted = df.sort_values("anchor_time").reset_index(drop=True)
n_test = int(len(df) * 0.15)

test_idx = df_sorted.index[-n_test:]
train_idx = df_sorted.index[:-n_test]

X_train = X.loc[train_idx]
y_train = y.loc[train_idx]
groups_train = groups.loc[train_idx]

X_test = X.loc[test_idx]
y_test = y.loc[test_idx]

# -----------------------------------------
# 5. GROUP-KFOLD OOF TRAINING
# -----------------------------------------

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(X_train))

models = []

for fold, (tr, val) in enumerate(gkf.split(X_train, y_train, groups_train)):
    print(f"\nFOLD {fold+1}")
    
    X_tr, X_val = X_train.iloc[tr], X_train.iloc[val]
    y_tr, y_val = y_train.iloc[tr], y_train.iloc[val]
    
    pos = (y_tr==1).sum()
    neg = (y_tr==0).sum()
    scale_pos_weight = neg / max(1, pos)
    
    clf = LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.03,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight
    )
    
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)]
    )
    
    oof[val] = clf.predict_proba(X_val)[:,1]
    joblib.dump(clf, os.path.join(OUT_DIR, f"lgb_churn_fold{fold+1}.pkl"))
    models.append(clf)

# -----------------------------------------
# 6. METRICS (OOF)
# -----------------------------------------

def topk_prec(y_true, y_score, pct=0.05):
    k = max(1, int(len(y_score) * pct))
    idx = np.argsort(y_score)[-k:]
    return y_true.iloc[idx].mean()

ap_oof = average_precision_score(y_train, oof)
roc_oof = roc_auc_score(y_train, oof)

print("\n=== OOF METRICS ===")
print("PR-AUC:", ap_oof)
print("ROC-AUC:", roc_oof)
print("Precision@5%:", topk_prec(y_train.reset_index(drop=True), pd.Series(oof), 0.05))
print("Precision@1%:", topk_prec(y_train.reset_index(drop=True), pd.Series(oof), 0.01))

# -----------------------------------------
# 7. FINAL MODEL → TRAIN FULL TRAIN, EVAL ON TEST
# -----------------------------------------

final_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    objective="binary",
    n_jobs=-1,
    random_state=42
)
final_model.fit(X_train, y_train)

test_proba = final_model.predict_proba(X_test)[:,1]
test_pred = (test_proba >= 0.5).astype(int)

ap_test = average_precision_score(y_test, test_proba)
roc_test = roc_auc_score(y_test, test_proba)

print("\n=== TEST METRICS ===")
print("PR-AUC:", ap_test)
print("ROC-AUC:", roc_test)
print("Precision@5%:", topk_prec(y_test.reset_index(drop=True), pd.Series(test_proba), 0.05))
print("Precision@1%:", topk_prec(y_test.reset_index(drop=True), pd.Series(test_proba), 0.01))

joblib.dump(final_model, os.path.join(OUT_DIR, "lgb_churn_final.pkl"))

print("\nDONE — All churn artifacts saved to:", OUT_DIR)


Loading: C:\Users\abhin\Desktop\Projects\FinTech Projects\Prophet\Dataset\synthetic_prophet_dataset_v2b.csv
(150000, 88)
    person         anchor_time  age gender  income became_member_on  \
0  U000000 2023-01-10 00:56:00   25      F   50588       2024-05-21   
1  U000000                 NaT   25      F   50588       2024-05-21   
2  U000000 2023-11-21 10:53:23   25      F   50588       2024-05-21   
3  U000000                 NaT   25      F   50588       2024-05-21   
4  U000000 2024-10-01 20:50:46   25      F   50588       2024-05-21   

  behavior_segment  activity_score loyalty_tier  treatment  ...  \
0           weekly        0.329376       bronze          0  ...   
1           weekly        0.329376       bronze          0  ...   
2           weekly        0.329376       bronze          0  ...   
3           weekly        0.329376       bronze          0  ...   
4           weekly        0.329376       bronze          0  ...   

   session_count_1d  txn_ewm_7d  txn_ewm_3d  burs